# 00 – Tải PKLot một lần và lưu thành Kaggle Dataset

Hugging Face giới hạn số lượt tải mỗi 5 phút. Tải ẩn danh trên Kaggle bị tính chung IP với người khác nên rất dễ gặp
lỗi **HTTP 429 (Rate limited)**. Notebook này tải dataset **một lần** rồi lưu thành Kaggle Dataset để các notebook sau
chỉ cần *Add Input*.

**Chuẩn bị:**
1. Tạo tài khoản miễn phí tại huggingface.co → *Settings → Access Tokens → Create new token* (loại **Read**).
2. Trên Kaggle: *Add-ons → Secrets → Add Secret*, tên `HF_TOKEN`, dán token, bật công tắc gắn vào notebook này.
3. *Settings → Internet* bật **On**.

**Sau khi chạy xong:** *Save Version* (Save & Run All) → tab *Output* → *New Dataset*, đặt tên ví dụ `pklot-yolo`.
Nếu bị ngắt giữa chừng, chạy lại cell tải: các file đã tải xong sẽ được bỏ qua.

In [ ]:
import json
import os
from pathlib import Path

HF_REPO = "dronefreak/PKLot"
SPLITS = ["train", "valid", "test"]
IMG_W, IMG_H = 1280, 720
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
DOWNLOAD_DIR = WORK_DIR / "pklot"   # lưu vào output để tạo Kaggle Dataset

def get_hf_token():
    # Có token thì giới hạn tải tính theo tài khoản, không bị chung IP với người khác trên Kaggle (lỗi 429)
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:   # chạy ngoài Kaggle
        return os.environ.get("HF_TOKEN")
    try:
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("Không đọc được secret HF_TOKEN:", repr(e))
        return os.environ.get("HF_TOKEN")

def write_yolo_labels(data_root: Path):
    # Tạo labels/*.txt từ metadata.jsonl (bbox COCO pixel → YOLO chuẩn hóa), khớp 100% file label gốc
    for split in SPLITS:
        out = data_root / "labels" / split
        out.mkdir(parents=True, exist_ok=True)
        for line in (data_root / "images" / split / "metadata.jsonl").read_text().splitlines():
            rec = json.loads(line)
            rows = [f"{c} {(x + w / 2) / IMG_W:.6f} {(y + h / 2) / IMG_H:.6f} {w / IMG_W:.6f} {h / IMG_H:.6f}"
                    for (x, y, w, h), c in zip(rec["objects"]["bbox"], rec["objects"]["categories"])]
            (out / f"{Path(rec['file_name']).stem}.txt").write_text("\n".join(rows) + "\n")

def download_pklot(dest: Path) -> Path:
    from huggingface_hub import snapshot_download   # cần bật Internet trong Settings
    token = get_hf_token()
    if not token:
        # Dừng ngay thay vì tải ẩn danh hàng giờ vì bị giới hạn (HTTP 429)
        raise RuntimeError(
            "Chưa có HF_TOKEN. Trên Kaggle: Add-ons → Secrets → thêm secret tên đúng 'HF_TOKEN' "
            "và ĐÁNH DẤU ô bên cạnh để gắn vào notebook này, rồi chạy lại. "
            "Hoặc Add Input dataset PKLot đã lưu để khỏi phải tải.")
    # Chỉ tải ảnh + metadata.jsonl; label tự tạo lại → giảm một nửa số request.
    # Chạy lại cell sẽ bỏ qua các file đã tải xong.
    snapshot_download(HF_REPO, repo_type="dataset", local_dir=dest, token=token, max_workers=8,
                      allow_patterns=["data/data.yaml", "data/images/*"])
    write_yolo_labels(dest / "data")
    (dest / "data" / ".complete").touch()
    return dest / "data"

def find_data_root() -> Path:
    # 1) Dataset đã gắn qua Add Input  2) đã tải xong trước đó  3) tải từ Hugging Face
    for depth in range(1, 6):   # không dùng "**" để tránh duyệt hết hàng chục nghìn ảnh
        for yaml_path in Path("/kaggle/input").glob("*/" * depth + "data.yaml"):
            if (yaml_path.parent / "labels").is_dir():
                return yaml_path.parent
    if (DOWNLOAD_DIR / "data" / ".complete").exists():
        return DOWNLOAD_DIR / "data"
    return download_pklot(DOWNLOAD_DIR)

DATA_ROOT = find_data_root()
print("DATA_ROOT =", DATA_ROOT)

## Kiểm tra đủ dữ liệu

In [ ]:
EXPECTED = {"train": (8346, 466306), "valid": (1981, 104014), "test": (2089, 123435)}   # theo trang dataset

for split, (n_img, n_box) in EXPECTED.items():
    imgs = len(list((DATA_ROOT / "images" / split).glob("*.jpg")))
    labels = list((DATA_ROOT / "labels" / split).glob("*.txt"))
    boxes = sum(len(p.read_text().split()) // 5 for p in labels)
    status = "OK" if (imgs, len(labels), boxes) == (n_img, n_img, n_box) else "THIẾU"
    print(f"{split:5s}: {imgs:5d} ảnh, {len(labels):5d} label, {boxes:7d} bbox  → {status}")